In [ ]:
# ==============================================================================
# ANALYSIS 1: CATEGORY VOLUME MATRIX
# Generates a cross-tabulation of working categories by Source System 
# to evaluate the distribution and volume of religion-related concepts.
# ==============================================================================

import os
import pandas as pd

print("\n" + "="*80)
print(" CATEGORY VOLUME BY SOURCE SYSTEM ")
print("="*80)

# --- 1. Setup & Load Data ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
processed_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "processed"))
final_app_file = os.path.join(processed_data_dir, "ontology_app_dataset.csv")

try:
    df = pd.read_csv(final_app_file, dtype={'CURIE': str})
except FileNotFoundError:
    print(f"[!] ERROR: Could not find {final_app_file}. Ensure Phase 3 has been run.")
    exit(1)

# --- 2. Clean and Standardize ---
if 'working_category' in df.columns:
    df['working_category'] = df['working_category'].fillna('Uncategorized').astype(str).str.strip().str.lower()
else:
    print("[!] ERROR: 'working_category' column missing from dataset.")
    exit(1)

# --- 3. Generate Cross-Tabulation ---
volume_matrix = pd.crosstab(
    index=df['Source_System'], 
    columns=df['working_category'], 
    margins=True, 
    margins_name='Total'
)

volume_matrix = volume_matrix.fillna(0).astype(int)

# --- 4. Custom Sorting ---
# Sort ROWS alphabetically, then move 'Total' to the bottom
volume_matrix = volume_matrix.sort_index()
if 'Total' in volume_matrix.index:
    total_row = volume_matrix.loc[['Total']]
    volume_matrix = pd.concat([volume_matrix.drop('Total'), total_row])

# Sort COLUMNS alphabetically, then move 'non-religious' and 'Total' to the end
cols = [c for c in volume_matrix.columns if c not in ['Total', 'non-religious']]
cols.sort()

if 'non-religious' in volume_matrix.columns:
    cols.append('non-religious')
if 'Total' in volume_matrix.columns:
    cols.append('Total')

# Apply the new column order
volume_matrix = volume_matrix[cols]

# --- 5. Display Results ---
print(volume_matrix.to_string())
print("="*80)

In [ ]:
import os
import pandas as pd

# --- Setup Directories ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
processed_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "processed"))
interim_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "interim"))

final_app_file = os.path.join(processed_data_dir, "ontology_app_dataset.csv")
output_file = os.path.join(interim_data_dir, "temp_sample_30.csv")

# --- Load and Sample ---
try:
    print(f"[*] Loading dataset from {final_app_file}...")
    # Enforce string typing for CURIE and Parent_IDs to prevent data mangling
    df = pd.read_csv(final_app_file, dtype={'CURIE': str, 'Parent_IDs': str, 'Concept_ID': str})
    
    print("[*] Drawing a random sample of 30 rows...")
    # random_state=None means you get a different 30 rows every time you run it
    sample_df = df.sample(n=30, random_state=None)
    
    sample_df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"[COMPLETE] Sample saved to: {output_file}")
    
except FileNotFoundError:
    print(f"[!] ERROR: Could not find {final_app_file}.")